# rBio circuit-class probe (K562)

Probe **rBio-1** on K562 perturbation outcome questions for genes stratified by GLMP circuit class
(Class I = feed-forward / no known feedback; Class III = bistable / positive feedback / mutual repression).

**Hypothesis:** rBio's reasoning quality and confidence (token-level `p(yes)`) differ between Class I
and Class III genes. Class III genes carry hysteresis / threshold dynamics that VCM verifiers struggle to
capture, so we expect either lower accuracy, more hedging, or systematically biased `p(yes)` on Class III.

**Design**
- **Cohorts:** all 17 Class III genes; 35 Class I genes (accuracy-matched on `mean_pearson` over the 14 benchmark methods, then random fill).
- **Toggle `USE_TOP_DE`** (config cell): controls whether the top-DE panel runs.
  - **`USE_TOP_DE = False` (default):** marker panel only — for each perturbation, one question **per symbol in `K562_MARKERS`**, skipping `target == perturbed` (so usually **5** questions/gene). **258** K562 rows (exact: `35×5 + 17×5 − 2` because **`GATA1`** and **`MYC`** appear in both the cohort and the marker panel) plus **50** rpe1 sanity (**10 × 5**; sanity genes are not in the marker panel) ⇒ **308** rows in `rbio_class_probe_results.parquet` (~40–55 min GPU, order-of-magnitude).
  - **`USE_TOP_DE = True`:** adds 20 top-DE-derived questions per gene — ~1,350 questions total, ~1.5–3 h on L4. Requires `top20_de_genes_per_perturbation.json` (or the K562 h5ad to recompute) on Drive.
- **Questions per gene:**
  - **B-1 — top-DE panel** *(only when `USE_TOP_DE=True`)*: 20 questions per gene targeting the experimental top-20 DE genes from K562 Perturb-seq (Replogle 2022).
  - **B-3 — marker panel** *(always on)*: up to **one question per marker** in (`GATA1, HBE1, HBG1, MYC, BCL11A`), excluding pairs where **`target`** equals the perturbation (**5**, or **4** if that gene is itself a marker).
- **Confidence:** token-level `p(yes)` extracted from `model.generate(..., output_scores=True)` at the `<answer>` span (Decision A-1).
- **rpe1 sanity:** **five** Class III + **five** Class I genes (frozen lists = first five symbols **sorted alphabetically** within each cohort after cohort construction) × marker panel, asked in `rpe1` cells (rBio's demonstrably trained domain). *Supplement note:* when Class III lists are alphabetical, the sanity cohort tends to emphasize **cell-cycle / checkpoint-style** bistable circuits (`BCL2L1`, `BUB1`, …), not necessarily the same **hematopoietic master regulators** highlighted in K562 headline panels.
- **Variant:** set `RBIO_VARIANT` in the config cell (e.g. `rbio1-TF` or `rbio1-EXP`).

**Inputs (mount via Drive)**
- `gene_circuit_classes.tsv` (workspace root) — *required*
- `results/merged_scores.tsv` — *required*
- `results/top20_de_genes_per_perturbation.json` — *required only when `USE_TOP_DE=True`* (produced by `k562-empirical-sequel/scripts/compute_de_genes.py`)
- `benchmark_data/ReplogleWeissman2022_K562_essential.h5ad` — *only needed if `USE_TOP_DE=True` and the JSON above is missing*

**Outputs (`/content/rbio_class_probe_results/<RBIO_VARIANT>/`)**
- `questions.parquet`
- `rbio_class_probe_results.parquet` (one row per question with answer, `p_yes`, `p_no`, `think`; `reflection` column kept for schema compatibility but **not analyzed** — `rbio1-TF` rarely emits `<reflection>`; check raw column for `rbio1-EXP`.)
- `per_gene_top_de.csv`, `per_gene_marker.csv`, `per_gene_marker_full.csv`
- `class_comparison_stats.csv`, `class_comparison_bootstrap.csv`, `class_comparison.png`
- `qualitative_audit.csv`


## 1. Setup: clone rBio, install deps, mount Drive, GPU check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.isdir('rbio-experiment'):
    !git clone https://github.com/czi-ai/rbio rbio-experiment

# Install rBio + our extras in a SINGLE resolver pass so rBio's `numpy<2` pin wins.
# anndata<0.11 and pyarrow<18 are the last versions that still support numpy 1.x.
!pip install -q -r rbio-experiment/requirements.txt \
    'anndata<0.11' 'pyarrow<18' scikit-learn matplotlib seaborn

# Verify the numpy pin actually took effect (avoids the `_center` ABI crash).
import subprocess, sys
out = subprocess.check_output([sys.executable, '-c', 'import numpy; print(numpy.__version__)']).decode().strip()
print(f'numpy version: {out}')
if not out.startswith('1.'):
    print('WARNING: numpy is not 1.x — forcing reinstall to numpy<2; you MUST restart the runtime after this cell.')
    !pip install -q --force-reinstall --no-deps 'numpy<2'
    raise SystemExit('Numpy was wrong version. Restart runtime: Runtime → Restart session, then run again.')

import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'CUDA: {p.name}, {p.total_memory/1e9:.1f} GB VRAM')
else:
    print('No GPU — inference will be very slow.')

## 2. Imports, paths, experiment config

Edit `DRIVE_ROOT` to point at the directory in your Drive that holds `gene_circuit_classes.tsv` and `results/`.

In [ ]:
import os, sys, re, json, time, shutil
from pathlib import Path

sys.path.insert(0, 'rbio-experiment')
from inference import load_model_and_tokenizer, download_sharded_checkpoint_from_s3

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm

# ---- paths (edit DRIVE_ROOT to match your layout) ----
DRIVE_ROOT          = Path('/content/drive/MyDrive/glmp')
GENE_CLASSES_TSV    = DRIVE_ROOT / 'gene_circuit_classes.tsv'
MERGED_SCORES_TSV   = DRIVE_ROOT / 'results' / 'merged_scores.tsv'
TOP20_DE_JSON       = DRIVE_ROOT / 'results' / 'top20_de_genes_per_perturbation.json'
H5AD_PATH           = DRIVE_ROOT / 'benchmark_data' / 'ReplogleWeissman2022_K562_essential.h5ad'

# ---- experiment config ----
RBIO_VARIANT     = 'rbio1-EXP'
S3_BUCKET        = 'czi-rbio'
BASE_MODEL       = 'Qwen/Qwen2.5-3B-Instruct'
N_CLASS_I        = 35
N_TOP_DE         = 20
RANDOM_SEED      = 42
CHECKPOINT_EVERY = 50

# Toggle the top-DE panel. False = marker panel only (258 K562 + 50 rpe1 ≈ 308 q).
# True = adds 20 top-DE-derived questions per gene (~1,350 q, ~1.5–3 h on L4)
# and requires top20_de_genes_per_perturbation.json (or the K562 h5ad) on Drive.
USE_TOP_DE = False

# Fresh Colab run: delete old parquet/checkpoints for this variant so you never resume stale rows.
# Set False only if you deliberately want to continue an interrupted run in the same VM session.
CLEAR_RESULTS_DIR_BEFORE_RUN = True

RESULTS_DIR = Path('/content/rbio_class_probe_results') / RBIO_VARIANT
if CLEAR_RESULTS_DIR_BEFORE_RUN and RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
    print(f'Removed prior results: {RESULTS_DIR}')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'RESULTS_DIR ready: {RESULTS_DIR}')

K562_MARKERS = ['GATA1', 'HBE1', 'HBG1', 'MYC', 'BCL11A']

SYSTEM_PROMPT_CoT = (
    'A conversation between User and Biologist. The user asks a question, '
    'and the Biologist solves it. The biologist first thinks about the reasoning process in the mind and '
    'then provides the user with the answer. The reasoning process and answer are enclosed within '
    '<think> </think> and <answer> </answer> tags, respectively, i.e., '
    '<think> reasoning process here </think> <answer> answer here </answer>. '
    'The Biologist provides the reasoning step-by-step.'
)

for p in (GENE_CLASSES_TSV, MERGED_SCORES_TSV):
    assert p.exists(), f'Missing: {p}'
print(f'Paths OK. USE_TOP_DE={USE_TOP_DE}. CLEAR_RESULTS_DIR_BEFORE_RUN={CLEAR_RESULTS_DIR_BEFORE_RUN}.')

## 3. Load rBio checkpoint (`RBIO_VARIANT`) and cache yes/no token IDs

First run downloads ~6 GB of sharded weights from `s3://czi-rbio/<RBIO_VARIANT>/` (see config cell).


In [ ]:
# Needs §2 (config) already executed for RBIO_VARIANT, AWS download helpers, etc.
import os
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
local_dir = f'model_weights/{RBIO_VARIANT}/'
if not os.path.isdir(local_dir):
    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    download_sharded_checkpoint_from_s3(s3, S3_BUCKET, f'{RBIO_VARIANT}/', local_dir)
model, tokenizer = load_model_and_tokenizer(BASE_MODEL, local_dir, device)
print(f'Loaded {RBIO_VARIANT} on {device}')

# Cache yes/no token-id sets so we can compute p(yes), p(no) at the answer span.
# We aggregate over leading-space + capitalized variants because the tokenizer is BPE.
def _ids(words):
    out = set()
    for w in words:
        for v in (w, ' ' + w, w.capitalize(), ' ' + w.capitalize(), w.upper(), ' ' + w.upper()):
            ids = tokenizer.encode(v, add_special_tokens=False)
            if len(ids) == 1:
                out.add(ids[0])
    return sorted(out)

YES_IDS = _ids(['yes'])
NO_IDS  = _ids(['no'])
print(f'yes ids: {YES_IDS}')
print(f'no  ids: {NO_IDS}')

## 4. Build cohorts: Class III (all 17) + accuracy-matched Class I (N=35)

**Gene table dedup:** if `gene_circuit_classes.tsv` lists the same symbol twice (e.g. TRRUST Class I + literature Class III), we keep **one row per gene** by preferring **higher-feedback tiers** (`III > II > IV > V > I`), then **`confidence`**, then **`literature` in `evidence_source`** — so cohorts match the curated circuit claim rather than topology-only stubs.

For each Class III gene, take the nearest-neighbour Class I gene by `mean_pearson` (averaged across all 14 benchmark methods), then random-fill the Class I cohort up to `N_CLASS_I=35` to give statistical headroom.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

classes_raw = pd.read_csv(GENE_CLASSES_TSV, sep='\t')
# One gene -> one class (duplicate HGNC rows exist in raw GLMP exports).
# Prefer curated feedback/bistability over TRRUST-only feed-forward rows.
TIER = {'III': 5, 'II': 4, 'IV': 3, 'V': 2, 'I': 1}
CONF = {'high': 3, 'medium': 2, 'low': 1}
_c = classes_raw.copy()
_c['_tier'] = _c['circuit_class'].map(TIER).fillna(0).astype(int)
_c['_conf'] = _c['confidence'].astype(str).str.lower().map(CONF).fillna(0).astype(int)
_c['_lit'] = _c['evidence_source'].fillna('').str.contains('literature', case=False).astype(int)
_c = _c.sort_values(['gene', '_tier', '_conf', '_lit'], ascending=[True, False, False, False])
classes = _c.groupby('gene', as_index=False).first().drop(columns=['_tier', '_conf', '_lit'])
print(f'Deduplicated gene table: {len(classes_raw)} -> {len(classes)} rows ({len(classes_raw) - len(classes)} duplicates collapsed)')
print('Class counts:', classes['circuit_class'].value_counts().to_dict())

ms = pd.read_csv(MERGED_SCORES_TSV, sep='\t')
gene_acc = ms.groupby('gene')['pearson_correlation'].mean().rename('mean_pearson')

class_iii = (classes[classes['circuit_class'] == 'III']
             .merge(gene_acc, on='gene', how='left'))
class_i_pool = (classes[classes['circuit_class'] == 'I']
                .merge(gene_acc, on='gene', how='left')
                .dropna(subset=['mean_pearson']))
print(f'Class III: {len(class_iii)} genes (with mean_pearson available: {class_iii["mean_pearson"].notna().sum()})')
print(f'Class I pool with mean_pearson: {len(class_i_pool)}')

chosen = []
remaining = class_i_pool.copy()
for _, row in class_iii.iterrows():
    if pd.isna(row['mean_pearson']) or remaining.empty:
        continue
    d = (remaining['mean_pearson'] - row['mean_pearson']).abs()
    pick = remaining.loc[d.idxmin(), 'gene']
    chosen.append(pick)
    remaining = remaining[remaining['gene'] != pick]

n_more = max(0, N_CLASS_I - len(chosen))
if n_more > 0 and len(remaining) > 0:
    fill = remaining.sample(n=min(n_more, len(remaining)), random_state=RANDOM_SEED)['gene'].tolist()
    chosen.extend(fill)

class_i = class_i_pool[class_i_pool['gene'].isin(chosen)].copy()
print(f'Selected Class I cohort: {len(class_i)}')
print('mean_pearson — Class III:', class_iii['mean_pearson'].describe()[['mean','std','50%']].to_dict())
print('mean_pearson — Class I  :', class_i['mean_pearson'].describe()[['mean','std','50%']].to_dict())

## 5. Top-20 DE per K562 perturbation *(only when `USE_TOP_DE=True`)*

Reads the precomputed JSON if present; otherwise computes inline from the Replogle 2022 K562 AnnData on Drive (~10 min on Colab Pro). Skipped entirely when `USE_TOP_DE=False`.

In [ ]:
top20_de = {}
if not USE_TOP_DE:
    print('USE_TOP_DE=False — skipping top-DE panel; will run marker panel only.')
elif TOP20_DE_JSON.exists():
    with open(TOP20_DE_JSON) as f:
        top20_de = json.load(f)
    print(f'Loaded precomputed top-20 DE for {len(top20_de)} perturbations from {TOP20_DE_JSON}')
else:
    assert H5AD_PATH.exists(), (
        f'USE_TOP_DE=True but neither {TOP20_DE_JSON} nor {H5AD_PATH} exists. '
        f'Either set USE_TOP_DE=False, run k562-empirical-sequel/scripts/compute_de_genes.py to produce the JSON, '
        f'or upload the K562 h5ad to Drive.'
    )
    import anndata as ad
    from scipy import sparse
    print('Computing top-20 DE inline...')
    adata = ad.read_h5ad(H5AD_PATH)
    print(f'  {adata.shape[0]} cells × {adata.shape[1]} genes')
    ctrl_mask = adata.obs['gene'].isin(['non-targeting']).values
    Xc = adata[ctrl_mask].X
    if sparse.issparse(Xc):
        Xc = Xc.toarray()
    Xc = Xc.astype(np.float64)
    lib_c = Xc.sum(1, keepdims=True); lib_c[lib_c == 0] = 1
    median_lib = np.median(lib_c)
    Xc_norm = np.log1p(Xc / lib_c * median_lib)
    ctrl_mean = Xc_norm.mean(0)
    gene_names = list(adata.var_names)
    perts = [p for p in adata.obs['gene'].unique() if p != 'non-targeting']
    top20_de = {}
    for p in tqdm(perts):
        m = (adata.obs['gene'] == p).values
        if m.sum() < 5:
            continue
        Xp = adata[m].X
        if sparse.issparse(Xp):
            Xp = Xp.toarray()
        Xp = Xp.astype(np.float64)
        lib_p = Xp.sum(1, keepdims=True); lib_p[lib_p == 0] = 1
        Xp_norm = np.log1p(Xp / lib_p * median_lib)
        delta = Xp_norm.mean(0) - ctrl_mean
        top = np.argsort(np.abs(delta))[-N_TOP_DE:][::-1]
        top20_de[p] = [gene_names[j] for j in top]
    TOP20_DE_JSON.parent.mkdir(parents=True, exist_ok=True)
    with open(TOP20_DE_JSON, 'w') as f:
        json.dump(top20_de, f)
    print(f'Saved {len(top20_de)} perturbation DE lists to {TOP20_DE_JSON}')

if USE_TOP_DE:
    covered_iii = sum(1 for g in class_iii['gene'] if g in top20_de)
    covered_i = sum(1 for g in class_i['gene'] if g in top20_de)
    print(f'DE coverage: Class III {covered_iii}/{len(class_iii)}, Class I {covered_i}/{len(class_i)}')

## 6. Build the question table

Per gene, generate `top_de` questions (B-1) and `marker_panel` questions (B-3).

**rpe1 sanity genes are frozen for reproducibility:** after `class_iii` / `class_i` are built, take the alphabetically-first **five** HGNC symbols in each cohort (`RPE1_SANITY_N = 5`). Edit `RPE1_SANITY_N` or replace with explicit lists if you need a predetermined panel.

**Interpretation (supplement wording):** *The rpe1 sanity cohort comprised the five alphabetically first Class III genes in each cohort, which for Class III were predominantly cell-cycle bistable circuits rather than hematopoietic master regulators.* rpe1 is mainly a trained-domain template check vs the K562 panels.

The code cell prints the exact gene lists for supplements.

In [ ]:
def make_question(perturbed, target, cell_line):
    return (f'Is a knockdown of {perturbed} in {cell_line} cells likely '
            f'to result in differential expression of {target}? The answer is either yes or no.')

rows = []
for cohort_name, cohort_df in [('class_iii_K562', class_iii), ('class_i_K562', class_i)]:
    for _, g in cohort_df.iterrows():
        gene = g['gene']
        meta = dict(
            cohort=cohort_name, perturbed=gene, cell_line='K562',
            circuit_class=g['circuit_class'], topology_type=g['topology_type'],
            mean_pearson=g['mean_pearson'],
        )
        if USE_TOP_DE:
            for target in top20_de.get(gene, [])[:N_TOP_DE]:
                if target == gene:
                    continue
                rows.append({**meta, 'target': target, 'question_type': 'top_de'})
        for target in K562_MARKERS:
            if target == gene:
                continue
            rows.append({**meta, 'target': target, 'question_type': 'marker_panel'})

# ---- rpe1 sanity cohort (frozen, reproducible) ----
RPE1_SANITY_N = 5
rpe1_iii = sorted(class_iii['gene'].tolist())[:RPE1_SANITY_N]
rpe1_i = sorted(class_i['gene'].tolist())[:RPE1_SANITY_N]
print('rpe1 sanity Class III genes:', rpe1_iii)
print('rpe1 sanity Class I genes  :', rpe1_i)

for gene in rpe1_iii + rpe1_i:
    is_iii = gene in set(class_iii['gene'])
    cls = 'III' if is_iii else 'I'
    cohort = 'class_iii_rpe1_sanity' if is_iii else 'class_i_rpe1_sanity'
    src = class_iii if is_iii else class_i
    topo = src.loc[src['gene'] == gene, 'topology_type'].iloc[0]
    for target in K562_MARKERS:
        if target == gene:
            continue
        rows.append(dict(
            cohort=cohort, perturbed=gene, cell_line='rpe1',
            circuit_class=cls, topology_type=topo, mean_pearson=np.nan,
            target=target, question_type='marker_panel',
        ))

questions_df = pd.DataFrame(rows)
questions_df['question'] = questions_df.apply(
    lambda r: make_question(r['perturbed'], r['target'], r['cell_line']), axis=1
)
questions_df['qid'] = questions_df.index.astype(str)
questions_df.to_parquet(RESULTS_DIR / 'questions.parquet')
print(f'Total questions: {len(questions_df)}')
print(questions_df.groupby(['cohort', 'question_type']).size())

## 7. `ask_rbio_with_logprobs` — inference with token-level p(yes), p(no)

Wraps rBio's prompt template but adds `output_scores=True` so we can read the softmax over `[yes_ids, no_ids]` at the position where the `<answer>` span begins. This is Decision A-1: a deterministic, single-forward-pass confidence estimate.

In [ ]:
ANSWER_RE = re.compile(r'<answer>\s*(.*?)\s*</answer>', flags=re.DOTALL)
THINK_RE  = re.compile(r'<think>(.*?)</think>',         flags=re.DOTALL)
REFL_RE   = re.compile(r'<reflection>(.*?)</reflection>', flags=re.DOTALL)

@torch.no_grad()
def ask_rbio_with_logprobs(question, system_prompt=SYSTEM_PROMPT_CoT, max_new_tokens=1024):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', padding=True).to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False, temperature=0.0, top_p=1.0, top_k=None,
        output_scores=True, return_dict_in_generate=True,
    )
    gen_ids = out.sequences[0, inputs.input_ids.shape[1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)

    p_yes = p_no = float('nan')
    am = ANSWER_RE.search(response)
    answer_text = am.group(1).strip().lower() if am else ''
    try:
        if am is not None:
            # Find the token position corresponding to the first character of the answer.
            char_pos = am.start(1)
            prefix = response[:char_pos]
            prefix_ids = tokenizer(prefix, add_special_tokens=False).input_ids
            tok_idx = len(prefix_ids)
            if 0 <= tok_idx < len(out.scores):
                logits = out.scores[tok_idx][0]
                probs = torch.softmax(logits.float(), dim=-1)
                p_yes = float(probs[YES_IDS].sum().item())
                p_no  = float(probs[NO_IDS].sum().item())
    except Exception:
        pass

    think = THINK_RE.findall(response)
    refl  = REFL_RE.findall(response)
    think_text = think[0].strip() if think else ''
    refl_text  = refl[0].strip()  if refl  else ''
    return {
        'answer':            answer_text,
        'p_yes':             p_yes,
        'p_no':              p_no,
        'think':             think_text,
        'reflection':        refl_text,
        'raw':               response,
        'n_think_tokens':    len(tokenizer.encode(think_text, add_special_tokens=False)) if think_text else 0,
        'n_reflect_tokens':  len(tokenizer.encode(refl_text,  add_special_tokens=False)) if refl_text  else 0,
    }

# Smoke test on rBio's own example question
smoke = ask_rbio_with_logprobs(
    'Is a knockdown of CPAMD8 in rpe1 cells likely to result in differential expression of SPARC? The answer is either yes or no.'
)
print('answer=%r  p_yes=%.3f  p_no=%.3f' % (smoke['answer'], smoke['p_yes'], smoke['p_no']))
print('think (first 300 chars):', smoke['think'][:300])

## 8. Run inference with checkpointing

Resumes from `rbio_class_probe_results.parquet` if present. Checkpoints every `CHECKPOINT_EVERY` questions so a Colab disconnect costs at most that many.

In [ ]:
ckpt_path = RESULTS_DIR / 'rbio_class_probe_results.parquet'
if ckpt_path.exists():
    done_df = pd.read_parquet(ckpt_path)
    done_qids = set(done_df['qid'])
    print(f'Resuming: {len(done_df)} questions already done.')
else:
    done_df = pd.DataFrame()
    done_qids = set()

records = done_df.to_dict('records') if len(done_df) else []
todo = questions_df[~questions_df['qid'].isin(done_qids)].reset_index(drop=True)
print(f'To run: {len(todo)} questions')

t0 = time.time()
for i, row in tqdm(todo.iterrows(), total=len(todo)):
    try:
        result = ask_rbio_with_logprobs(row['question'])
    except Exception as e:
        result = dict(answer='', p_yes=float('nan'), p_no=float('nan'),
                      think='', reflection='', raw=f'ERROR: {e}',
                      n_think_tokens=0, n_reflect_tokens=0)
    rec = {**row.to_dict(), **result, 'rbio_variant': RBIO_VARIANT}
    records.append(rec)
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(records).to_parquet(ckpt_path)
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(todo) - (i + 1)) / max(rate, 1e-6)
        print(f'  ckpt @ {i+1}/{len(todo)}  rate={rate:.2f} q/s  eta={eta/60:.1f} min')

result_df = pd.DataFrame(records)
result_df.to_parquet(ckpt_path)
print(f'Done: {len(result_df)} rows in {(time.time()-t0)/60:.1f} min')

## 9. Per-gene aggregation

In [ ]:
result_df = pd.read_parquet(RESULTS_DIR / 'rbio_class_probe_results.parquet')

def agg_gene(df):
    return pd.Series({
        'n_questions':         len(df),
        'p_yes_mean':          df['p_yes'].mean(),
        'p_yes_std':           df['p_yes'].std(),
        'frac_yes_decision':   (df['answer'].str.lower() == 'yes').mean(),
        'mean_think_tokens':   df['n_think_tokens'].mean(),
    })

top_de_rows = result_df[(result_df['question_type'] == 'top_de') & (result_df['cell_line'] == 'K562')]
if len(top_de_rows):
    per_gene_top_de = (
        top_de_rows
        .groupby(['cohort', 'perturbed', 'circuit_class', 'topology_type', 'mean_pearson'], dropna=False)
        .apply(agg_gene)
        .reset_index()
    )
else:
    per_gene_top_de = pd.DataFrame(
        columns=['cohort', 'perturbed', 'circuit_class', 'topology_type', 'mean_pearson',
                 'n_questions', 'p_yes_mean', 'p_yes_std', 'frac_yes_decision',
                 'mean_think_tokens']
    )
per_gene_marker = (
    result_df[(result_df['question_type'] == 'marker_panel') & (result_df['cell_line'] == 'K562')]
    .groupby(['cohort', 'perturbed', 'circuit_class', 'topology_type'], dropna=False)
    .apply(agg_gene)
    .reset_index()
)
# Table S1-friendly: merge SOTA `mean_pearson` (from merged_scores) alongside rBio per-gene aggregates
per_gene_marker_full = (
    per_gene_marker
    .rename(columns={'perturbed': 'gene'})
    .merge(gene_acc.rename('mean_pearson_sota').rename_axis('gene').reset_index(), on='gene', how='left')
    [[
        'gene', 'circuit_class', 'cohort', 'topology_type',
        'n_questions', 'frac_yes_decision', 'p_yes_mean', 'mean_think_tokens',
        'mean_pearson_sota',
    ]]
    .sort_values(['circuit_class', 'gene'])
)
per_gene_top_de.to_csv(RESULTS_DIR / 'per_gene_top_de.csv', index=False)
per_gene_marker.to_csv(RESULTS_DIR / 'per_gene_marker.csv', index=False)
per_gene_marker_full.to_csv(RESULTS_DIR / 'per_gene_marker_full.csv', index=False)
print(f'Per-gene rows — top-DE: {len(per_gene_top_de)},  marker: {len(per_gene_marker)}')
per_gene_marker.head()

## 10. Class I vs Class III statistical comparison + plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

def compare_class(df, metric, panel):
    iii = df[df['circuit_class'] == 'III'][metric].dropna()
    one = df[df['circuit_class'] == 'I'][metric].dropna()
    if len(iii) == 0 or len(one) == 0:
        return None
    u, p = mannwhitneyu(iii, one, alternative='two-sided')
    return dict(panel=panel, metric=metric,
                n_iii=len(iii), n_i=len(one),
                median_iii=iii.median(), median_i=one.median(),
                U=float(u), p=float(p))

rows = []
for metric in ['frac_yes_decision', 'p_yes_mean', 'mean_think_tokens']:
    for df, panel in [(per_gene_marker, 'marker'), (per_gene_top_de, 'top_de')]:
        r = compare_class(df, metric, panel)
        if r is not None:
            rows.append(r)
stats_df = pd.DataFrame(rows)
stats_df.to_csv(RESULTS_DIR / 'class_comparison_stats.csv', index=False)
print(stats_df.to_string(index=False))

# Bootstrap: median difference (III - I) for frac_yes_decision on marker panel (PRIMARY endpoint)
rng_boot = np.random.default_rng(RANDOM_SEED)
B = 10_000
m = per_gene_marker[per_gene_marker['circuit_class'].isin(['I', 'III'])].copy()
iii_v = m[m['circuit_class'] == 'III']['frac_yes_decision'].to_numpy(dtype=float)
i_v = m[m['circuit_class'] == 'I']['frac_yes_decision'].to_numpy(dtype=float)
boot_diff = np.empty(B)
for b in range(B):
    samp_iii = rng_boot.choice(iii_v, size=len(iii_v), replace=True)
    samp_i = rng_boot.choice(i_v, size=len(i_v), replace=True)
    boot_diff[b] = np.median(samp_iii) - np.median(samp_i)
ci_low, ci_high = np.percentile(boot_diff, [2.5, 97.5])
boot_row = pd.DataFrame([dict(
    panel='marker',
    metric='frac_yes_decision',
    statistic='median_III_minus_median_I',
    n_bootstrap=B,
    point_estimate=float(np.median(iii_v) - np.median(i_v)),
    ci_low=float(ci_low),
    ci_high=float(ci_high),
)])
boot_row.to_csv(RESULTS_DIR / 'class_comparison_bootstrap.csv', index=False)
print('\nBootstrap (marker, frac_yes_decision):')
print(boot_row.to_string(index=False))

panels = [('marker panel', per_gene_marker)]
if len(per_gene_top_de):
    panels.append(('top-DE panel', per_gene_top_de))
metrics = ['frac_yes_decision', 'p_yes_mean', 'mean_think_tokens']
fig, axes = plt.subplots(len(metrics), len(panels),
                         figsize=(5.5 * len(panels), 3.8 * len(metrics)),
                         squeeze=False)
for i, metric in enumerate(metrics):
    for j, (panel_name, panel_df) in enumerate(panels):
        ax = axes[i, j]
        sub = panel_df[panel_df['circuit_class'].isin(['I', 'III'])]
        sns.boxplot(data=sub, x='circuit_class', y=metric, order=['I', 'III'], ax=ax,
                    color='lightgrey')
        sns.stripplot(data=sub, x='circuit_class', y=metric, order=['I', 'III'], ax=ax,
                      color='black', alpha=0.6, size=3)
        ax.set_title(f'{metric} — {panel_name}')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'class_comparison.png', dpi=150)
plt.show()

# rpe1 sanity check: p(yes) should be a sane mid-range distribution, not collapsed.
rpe = result_df[result_df['cell_line'] == 'rpe1']
print('\nrpe1 sanity p(yes) summary:')
print(rpe.groupby('circuit_class')['p_yes'].describe())

## 11. Qualitative reasoning audit

Three Class III genes (`MYC`, `GATA1`, `BCL2L1`) and three Class I genes (highest `mean_pearson` in the cohort) — dump full `<think>` traces for the marker-panel questions for side-by-side reading.

In [ ]:
SHOWCASE_III = [g for g in ['MYC', 'GATA1', 'BCL2L1'] if g in set(class_iii['gene'])]
showcase_i = (
    per_gene_marker[per_gene_marker['circuit_class'] == 'I']
    .merge(class_i[['gene', 'mean_pearson']], left_on='perturbed', right_on='gene', how='left')
    .sort_values('mean_pearson', ascending=False)
    .head(3)['perturbed'].tolist()
)
print('Showcase Class III:', SHOWCASE_III)
print('Showcase Class I  :', showcase_i)

audit = result_df[
    result_df['perturbed'].isin(SHOWCASE_III + showcase_i)
    & (result_df['question_type'] == 'marker_panel')
    & (result_df['cell_line'] == 'K562')
].copy()

for gene in SHOWCASE_III + showcase_i:
    sub = audit[audit['perturbed'] == gene]
    if sub.empty:
        continue
    cls = sub['circuit_class'].iloc[0]
    print('\n' + '=' * 78)
    print(f'{gene}  (Class {cls})')
    print('=' * 78)
    for _, r in sub.iterrows():
        print('\nQ: ' + str(r['question']))
        print('  → answer=%r  p(yes)=%.3f' % (r['answer'], r['p_yes']))
        print('  THINK: ' + str(r['think'])[:600])
audit.to_csv(RESULTS_DIR / 'qualitative_audit.csv', index=False)
print('\nSaved audit to', RESULTS_DIR / 'qualitative_audit.csv')

## 12. Save final results back to Drive (optional)

In [ ]:
OUT_DRIVE = DRIVE_ROOT / 'results' / 'rbio_class_probe' / RBIO_VARIANT
OUT_DRIVE.mkdir(parents=True, exist_ok=True)
import shutil
for fn in ['questions.parquet', 'rbio_class_probe_results.parquet',
           'per_gene_top_de.csv', 'per_gene_marker.csv', 'per_gene_marker_full.csv',
           'class_comparison_stats.csv', 'class_comparison_bootstrap.csv',
           'class_comparison.png', 'qualitative_audit.csv']:
    src = RESULTS_DIR / fn
    if src.exists():
        shutil.copy(src, OUT_DRIVE / fn)
        print(f'  -> {OUT_DRIVE / fn}')
print('Saved.')